In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression  
from sklearn.metrics import mean_squared_error, r2_score,accuracy_score, classification_report, confusion_matrix
from autofeat import AutoFeatRegressor 
import featuretools as ft

In [ ]:
transform_pirmitive = ft.list_primitives()[ft.list_primitives()['type'] == 'transform'].head(10)


,name,type,description,valid_inputs,return_type
65,is_leap_year,transform,Determines the is_leap_year attribute of a dat...,<ColumnSchema (Logical Type = Datetime)>,<ColumnSchema (Logical Type = BooleanNullable)>
66,part_of_day,transform,Determines the part of day of a datetime.,<ColumnSchema (Logical Type = Datetime)>,<ColumnSchema (Logical Type = Categorical) (Se...
67,less_than,transform,Determines if values in one list are less than...,"<ColumnSchema (Logical Type = Datetime)>, <Col...",<ColumnSchema (Logical Type = BooleanNullable)>
68,absolute,transform,Computes the absolute value of a number.,<ColumnSchema (Semantic Tags = ['numeric'])>,<ColumnSchema (Semantic Tags = ['numeric'])>
69,expanding_max,transform,Computes the expanding maximum of events over ...,"<ColumnSchema (Semantic Tags = ['numeric'])>, ...",<ColumnSchema (Semantic Tags = ['numeric'])>
70,multiply_numeric_scalar,transform,Multiplies each element in the list by a scalar.,<ColumnSchema (Semantic Tags = ['numeric'])>,<ColumnSchema (Semantic Tags = ['numeric'])>
71,num_words,transform,Determines the number of words in a string. Wo...,<ColumnSchema (Logical Type = NaturalLanguage)>,<ColumnSchema (Logical Type = IntegerNullable)...
72,full_name_to_title,transform,Determines the title from a person's name.,<ColumnSchema (Logical Type = PersonFullName)>,<ColumnSchema (Logical Type = Categorical) (Se...
73,year,transform,Determines the year value of a datetime.,<ColumnSchema (Logical Type = Datetime)>,"<ColumnSchema (Logical Type = Ordinal: [1, 2, ..."
74,equal,transform,Determines if values in one list are equal to ...,<ColumnSchema>,<ColumnSchema (Logical Type = BooleanNullable)>


In [3]:
df = pd.read_csv('bostonHousing.csv')
df.head()

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,b,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


In [4]:
df.fillna( df.mode().iloc[0],inplace=True)

In [6]:
X = df.drop(columns=["medv"])  
y = df["medv"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Erreur quadratique moyenne (MSE) : {mse:.2f}")
print(f"Score R² : {r2:.2f}")

Erreur quadratique moyenne (MSE) : 24.77
Score R² : 0.66


In [7]:
#https://medium.com/@boukamchahamdi/autofeat-automating-feature-engineering-with-python-f22ec23265a9
af = AutoFeatRegressor( feateng_steps=2,n_jobs=-1)  

X_train_af = af.fit_transform(X_train, y_train)
X_test_af = af.transform(X_test)
X_train_af.head()
print(f"Nombre de nouvelles features créées : {X_train_af.shape[1] - X_train.shape[1]}")

model_af = LinearRegression()
model_af.fit(X_train_af, y_train)
y_pred_af = model_af.predict(X_test_af)

mse = mean_squared_error(y_test, y_pred_af)
r2 = r2_score(y_test, y_pred_af)
print("\n Performance APRÈS AutoFeat")
print(f"Erreur quadratique moyenne (MSE) : {mse:.2f}")
print(f"Score R² : {r2:.2f}")

c:\Users\H P\git\projetTutoreAutoFE\venv\lib\site-packages\autofeat\featsel.py:270: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  if np.max(np.abs(correlations[c].ravel()[:i])) < 0.9:


Nombre de nouvelles features créées : 37

 Performance APRÈS AutoFeat
Erreur quadratique moyenne (MSE) : 14.58
Score R² : 0.80


c:\Users\H P\git\projetTutoreAutoFE\venv\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [ ]:
#https://medium.com/@jimliu741523/comparsion-of-feature-engineering-tool-autofeat-featuretools-headjack-e8eef972fec9
trans_primitives=['add_numeric', 'subtract_numeric', 'multiply_numeric', 'divide_numeric'] 
es = ft.EntitySet(id="bostonHousing")
es = es.add_dataframe(
    dataframe_name="df", 
    dataframe=df, 
    index='index',
    make_index=True)

feature_matrix, feature_names = ft.dfs(entityset = es, 
     target_dataframe_name='df', 
     max_depth=1,    
     verbose=0,
      n_jobs = -1,
     trans_primitives=trans_primitives
) 